In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import json

# Reproducibility
tf.random.set_seed(42)
np.random.seed(42)

PROJECT_ROOT = Path.cwd().parent.parent if Path.cwd().name == 'step1_mmd_net' else Path.cwd()
print('Project root:', PROJECT_ROOT)

Project root: /Users/kirtan/Projects /NNDL


In [2]:
# 1. DATA LOADING (The 3000 PCA genes)

# Source data
X_train_src = np.load(PROJECT_ROOT / 'step3_X_train.npy').astype(np.float32)
y_train_src = np.load(PROJECT_ROOT / 'step3_y_train.npy').astype(np.int64)

X_test_src = np.load(PROJECT_ROOT / 'step3_X_test.npy').astype(np.float32)
y_test_src = np.load(PROJECT_ROOT / 'step3_y_test.npy').astype(np.int64)

# Load unlabeled Target data for MMD alignment
try:
    X_target_unlabeled = np.load(PROJECT_ROOT / 'improvements/2_batch_correction/batch_correction_output/gse126030_corrected_coral.npy').astype(np.float32)
    print("Found CORAL-aligned target data. We'll use this for MMD training.")
except:
    X_target_unlabeled = np.load(PROJECT_ROOT / 'gse126030_preprocessed.npy').astype(np.float32)
    print("Falling back to preprocessed but unaligned target data for MMD training.")

# Optional labeled target data for evaluation ONLY
X_target_eval = None
y_target_eval = None
labels_csv = PROJECT_ROOT / 'improvements/1_label_validation/label_validation_output/gse126030_reclustered_labels.csv'

if labels_csv.exists():
    df = pd.read_csv(labels_csv)
    mask = (df['confidence'] >= 0.55) & (df['new_class'] != 'Uncertain')
    
    with open(PROJECT_ROOT / 'step3_label_mapping.json') as f:
        label_map = json.load(f)
    class_names = [label_map[str(i)] for i in range(len(label_map))]
    
    X_target_eval = X_target_unlabeled[mask.values]
    y_target_eval = np.array([class_names.index(n) for n in df.loc[mask, 'new_class']], dtype=np.int64)

print(f"Source Train: {X_train_src.shape}")
print(f"Target Train (Unlabeled): {X_target_unlabeled.shape}")
if X_target_eval is not None:
    print(f"Target Test (Labeled subset): {X_target_eval.shape}")

Falling back to preprocessed but unaligned target data for MMD training.
Source Train: (6824, 3000)
Target Train (Unlabeled): (63877, 3000)
Target Test (Labeled subset): (12776, 3000)


# Step 1: Implement the Multiple Kernel MMD function

The core of this model is computing the difference between two distributions, $P(X_{source})$ and $Q(X_{target})$. We use an RBF function which compares every point in the source batch to every point in the target batch.

In [3]:
@tf.function
def compute_pairwise_distances(x, y):
    """Computes the squared pairwise Euclidean distances between x and y."""
    if not hasattr(compute_pairwise_distances, "is_compiled"):
        pass # Optional warmup hook
    x_norm = tf.reduce_sum(tf.square(x), 1)
    y_norm = tf.reduce_sum(tf.square(y), 1)

    x_y = tf.tensordot(x, y, axes=[[1], [1]])
    
    # ||x - y||^2 = ||x||^2 - 2x.y + ||y||^2
    dist = tf.expand_dims(x_norm, 1) - 2.0 * x_y + tf.expand_dims(y_norm, 0)

    # Ensure non-negative due to floating point inaccuracies
    return tf.maximum(dist, 0.0)

@tf.function
def compute_gaussian_kernel(x, y, sigmas=[1.0, 5.0, 10.0]):
    """Multi-scale RBF kernel."""
    beta = 1.0 / (2.0 * (tf.expand_dims(sigmas, 1)))

    dist = compute_pairwise_distances(x, y)
    s = tf.matmul(beta, tf.reshape(dist, (1, -1)))
    
    return tf.reshape(tf.reduce_sum(tf.exp(-s), 0), tf.shape(dist))

@tf.function
def compute_mmd(source_features, target_features, sigmas=[1.0, 5.0, 10.0]):
    """Calculates the Maximum Mean Discrepancy (MMD) between source and target."""
    
    # Kernel values
    k_xx = compute_gaussian_kernel(source_features, source_features, sigmas)
    k_yy = compute_gaussian_kernel(target_features, target_features, sigmas)
    k_xy = compute_gaussian_kernel(source_features, target_features, sigmas)
    
    # MMD formula: E[k(X,X)] + E[k(Y,Y)] - 2E[k(X,Y)]
    # We take the mean across the batch
    m_xx = tf.reduce_mean(k_xx)
    m_yy = tf.reduce_mean(k_yy)
    m_xy = tf.reduce_mean(k_xy)
    
    return m_xx + m_yy - 2 * m_xy

# Step 2: Build the Keras Model (Custom Train Step)

To process both datasets simultaneously, we override the `train_step`. The model natively trains on both datasets at the same time in parallel batches.

In [4]:
class MMDModel(keras.Model):
    def __init__(self, encoder, classifier, mmd_weight=1.0, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.classifier = classifier
        self.mmd_weight = mmd_weight
        self.loss_tracker = keras.metrics.Mean(name="loss")
        self.class_loss_tracker = keras.metrics.Mean(name="class_loss")
        self.mmd_loss_tracker = keras.metrics.Mean(name="mmd_loss")
        self.accuracy_tracker = keras.metrics.SparseCategoricalAccuracy(name="accuracy")

    def call(self, inputs, training=False):
        features = self.encoder(inputs, training=training)
        return self.classifier(features, training=training)

    def train_step(self, data):
        # We expect a merged dataset: (source_x, source_y), (target_x)
        (source_x, source_y), (target_x,) = data

        with tf.GradientTape() as tape:
            # 1. Encode both domains
            source_features = self.encoder(source_x, training=True)
            target_features = self.encoder(target_x, training=True)

            # 2. Supervised Classification (Source only)
            predictions = self.classifier(source_features, training=True)
            class_loss = self.compiled_loss(source_y, predictions)

            # 3. Unsupervised Domain Alignment (MMD)
            mmd_loss = compute_mmd(source_features, target_features)
            
            # Combine losses
            total_loss = class_loss + (self.mmd_weight * mmd_loss)

        # Apply gradients
        trainable_vars = self.encoder.trainable_variables + self.classifier.trainable_variables
        gradients = tape.gradient(total_loss, trainable_vars)
        self.optimizer.apply_gradients(zip(gradients, trainable_vars))

        # Update metrics
        self.loss_tracker.update_state(total_loss)
        self.class_loss_tracker.update_state(class_loss)
        self.mmd_loss_tracker.update_state(mmd_loss)
        self.accuracy_tracker.update_state(source_y, predictions)

        return {
            "loss": self.loss_tracker.result(),
            "class_loss": self.class_loss_tracker.result(),
            "mmd_loss": self.mmd_loss_tracker.result(),
            "accuracy": self.accuracy_tracker.result(),
        }
        
    @property
    def metrics(self):
        return [self.loss_tracker, self.class_loss_tracker, self.mmd_loss_tracker, self.accuracy_tracker]

print("Model scaffold ready. Run next cells to construct and train.")

Model scaffold ready. Run next cells to construct and train.


In [5]:
# 3. Model Initialization & Compilation
# The architecture needs a feature extractor and a classifier

input_shape = (X_train_src.shape[1],)
num_classes = len(class_names)

def build_feature_extractor(input_shape):
    inputs = keras.Input(shape=input_shape)
    x = layers.Dense(1024, activation='relu')(inputs)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(256, activation='relu', name='features')(x)
    return keras.Model(inputs, x, name='feature_extractor')

def build_classifier(feature_dim, num_classes):
    inputs = keras.Input(shape=(feature_dim,))
    x = layers.Dense(128, activation='relu')(inputs)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return keras.Model(inputs, outputs, name='classifier')

feature_extractor = build_feature_extractor(input_shape)
classifier = build_classifier(256, num_classes)

mmd_model = MMDModel(feature_extractor, classifier, mmd_weight=1.0)

mmd_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False)
)

print("MMD-Net models initialized and compiled.")

MMD-Net models initialized and compiled.


In [6]:
# 4. Data Loaders Formulation
BATCH_SIZE = 64

# Source Dataset yields (x, y)
source_dataset = tf.data.Dataset.from_tensor_slices((X_train_src, y_train_src))
source_dataset = source_dataset.shuffle(10000).batch(BATCH_SIZE, drop_remainder=True)

# Target Unlabeled Dataset yields (x,)
# Need to yield tuples to match the zip structure expected by train_step
target_dataset = tf.data.Dataset.from_tensor_slices((X_target_unlabeled,))
target_dataset = target_dataset.shuffle(10000).batch(BATCH_SIZE, drop_remainder=True)

# Ensure the datasets loop indefinitely so `zip` continues until the source is exhausted
target_dataset = target_dataset.repeat()

# Zip them together
# The zipped dataset yields ((source_batch_x, source_batch_y), (target_batch_x,))
train_dataset = tf.data.Dataset.zip((source_dataset, target_dataset))

# Validation target (for evaluation of Macro F1 on target)
# In standard evaluation, model.predict/evaluate only expects (x, y)
eval_dataset = tf.data.Dataset.from_tensor_slices((X_target_eval, y_target_eval)).batch(BATCH_SIZE)

print("Datasets mapped and zipped.")

Datasets mapped and zipped.


In [7]:
# 5. Training Loop
# Because target_dataset repeats indefinitely, steps_per_epoch is determined by source_dataset length.
EPOCHS = 30

history = mmd_model.fit(
    train_dataset,
    epochs=EPOCHS
)

# 6. Evaluation on Target Domain
from sklearn.metrics import classification_report, f1_score

# Predict on Target Eval
y_pred_probs = mmd_model.predict(X_target_eval)
y_pred = np.argmax(y_pred_probs, axis=1)

macro_f1 = f1_score(y_target_eval, y_pred, average='macro')
print(f"Target Macro F1 Score: {macro_f1:.4f}")
print("\nClassification Report on Target Domain:")
print(classification_report(y_target_eval, y_pred, target_names=class_names))

Epoch 1/30


/Users/kirtan/Projects /NNDL/.venv/lib/python3.11/site-packages/keras/src/backend/tensorflow/trainer.py:695: UserWarning: `model.compiled_loss()` is deprecated. Instead, use `model.compute_loss(x, y, y_pred, sample_weight, training)`.
  warnings.warn(


106/106 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.4909 - class_loss: 1.3383 - loss: 1.4329 - mmd_loss: 0.0946
Epoch 2/30
106/106 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.4909 - class_loss: 1.3383 - loss: 1.4329 - mmd_loss: 0.0946
Epoch 2/30
106/106 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.7584 - class_loss: 0.6539 - loss: 0.7477 - mmd_loss: 0.0938
Epoch 3/30
106/106 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.7584 - class_loss: 0.6539 - loss: 0.7477 - mmd_loss: 0.0938
Epoch 3/30
106/106 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.8557 - class_loss: 0.4034 - loss: 0.4972 - mmd_loss: 0.0938
Epoch 4/30
106/106 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.8557 - class_loss: 0.4034 - loss: 0.4972 - mmd_loss: 0.0938
Epoch 4/30
106/106 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9093 - class_loss: 0.2545 - loss: 0.3483 - mmd_loss: 0.0938
Epoch 5/30
106/106 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9093 - class_loss: 0.2545 - loss: 0.3483 - mmd_loss: 0.09

/Users/kirtan/Projects /NNDL/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/kirtan/Projects /NNDL/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/kirtan/Projects /NNDL/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.c